Under each cell there is a description of that cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Connect Google Drive to Colab

In [ ]:
import pandas as pd
import json


def load_json_sample(filename, n_lines=100000):
    data = []
    with open(filename, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n_lines:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

business = load_json_sample('/content/drive/MyDrive/FYP/Data/yelp_academic_dataset_business.json', n_lines=50000)
user = load_json_sample('/content/drive/MyDrive/FYP/Data/yelp_academic_dataset_user.json', n_lines=50000)
review = load_json_sample('/content/drive/MyDrive/FYP/Data/yelp_academic_dataset_review.json', n_lines=100000)

Loads a specified number of lines from each large Yelp dataset (user, review, and business) into memory-efficient pandas DataFrames.

In [ ]:
print(business.shape)
print(user.shape)
print(review.shape)

business.head()

Inspect Dataset Sizes
Prints the dimensions of the loaded business, user, and review DataFrames to verify content.

In [ ]:
active_users = user[user['review_count'] > 10]

sample_users = active_users.sample(n=10000, random_state=42)

review_sample = review[review['user_id'].isin(sample_users['user_id'])]

merged_df = review_sample.merge(business, on='business_id', how='left')

merged_df = merged_df.merge(user, on='user_id', suffixes=('_biz', '_user'))

Filter Active Users & Sample

Filters out users with fewer than 10 reviews and randomly selects a sample of 10,000 active users for analysis.

In [ ]:
user_summary = merged_df.groupby('user_id').agg({
    'stars_x': 'mean',
    'useful_user': 'sum',
    'cool_user': 'sum',
    'funny_user': 'sum',
    'review_id': 'count'
})


user_summary.columns = ['avg_rating', 'total_useful', 'total_cool', 'total_funny', 'review_count']
user_summary.reset_index(inplace=True)

 Aggregate User Behavior Features
Computes user-level features such as average rating, total helpful/cool/funny votes, and review count for each user.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns


X = user_summary[['avg_rating', 'total_useful', 'total_cool', 'total_funny', 'review_count']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)


plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inertia, marker='o')
plt.title('Elbow Method for Choosing Optimal Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

kmeans = KMeans(n_clusters=4, random_state=42)
user_summary['cluster'] = kmeans.fit_predict(X_scaled)


cluster_summary = user_summary.groupby('cluster').mean(numeric_only=True)
print(cluster_summary)


sample = user_summary.sample(500, random_state=1)

sns.pairplot(sample, vars=['avg_rating', 'total_useful', 'review_count'], hue='cluster', palette='Set2')
plt.suptitle('Customer Segments Based on Engagement and Reviews', y=1.02)
plt.show()

Apply K-Means Clustering
Segments users into behavioral clusters using the K-Means algorithm based on standardized engagement metrics.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X = user_summary[['avg_rating', 'total_useful', 'total_cool', 'total_funny', 'review_count']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
user_summary['cluster'] = kmeans.fit_predict(X_scaled)

Normalize Features

Standardizes all numeric features so that they contribute equally in distance-based models like K-Means and KNN.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cluster_means = user_summary.groupby('cluster').mean(numeric_only=True)

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, annot=True, cmap='coolwarm')
plt.title('Cluster Profile Heatmap')
plt.show()

Cluster Heatmap
Plots the average values of user features across each cluster to highlight behavioral differences.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


target = 'review_count'
features = ['avg_rating', 'total_useful', 'total_cool', 'total_funny']

X = user_summary[features]
y = user_summary[target]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

Train-Test Split & Model Training
Splits the data and trains a Random Forest model to predict user review count (CLV).

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

# Evaluate with MAE
Computes Mean Absolute Error to measure average prediction deviation from true values.



In [ ]:
from sklearn.ensemble import GradientBoostingRegressor


gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)


gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)
gb_mae = mean_absolute_error(y_test, gb_pred)

print(f"Gradient Boosting RMSE: {gb_rmse:.2f}")
print(f"Gradient Boosting R² Score: {gb_r2:.2f}")
print(f"Gradient Boosting MAE: {gb_mae:.2f}")

In [ ]:
cluster_profiles = user_summary.groupby('cluster').mean(numeric_only=True)

print(cluster_profiles[['avg_rating', 'total_useful', 'total_cool', 'total_funny', 'review_count']])

# Cluster Behavior Profiles
Computes the mean engagement behavior for each cluster to support interpretation.

In [ ]:
import pandas as pd

importances = model.feature_importances_
feature_names = X.columns

importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importance_df.plot(kind='barh')
plt.title("Feature Importance for Predicting CLV (Review Count)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

# Feature Importance
Visualizes how influential each feature is in predicting CLV using Random Forest.

In [ ]:
from sklearn.decomposition import PCA


pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

user_summary['pca_1'] = components[:, 0]
user_summary['pca_2'] = components[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=user_summary, x='pca_1', y='pca_2', hue='cluster', palette='Set2')
plt.title("Customer Segmentation Visualized with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()


# PCA for Visualization
Reduces dimensionality to 2D for easier visualization of user segments.

In [ ]:
!pip install xgboost
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor

features = ['avg_rating', 'total_useful', 'total_cool', 'total_funny']
target = 'review_count'


X = user_summary[features]
y = user_summary[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


knn = KNeighborsRegressor()
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(knn, param_grid_knn, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_knn.fit(X_train, y_train)
best_knn = grid_knn.best_estimator_
knn_pred = best_knn.predict(X_test)
knn_rmse = np.sqrt(mean_squared_error(y_test, knn_pred))
print("KNN Best Params:", grid_knn.best_params_)
print("KNN RMSE:", knn_rmse)


xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

grid_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_
xgb_pred = best_xgb.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
print("XGBoost Best Params:", grid_xgb.best_params_)
print("XGBoost RMSE:", xgb_rmse)

dt_model = DecisionTreeRegressor(random_state=42)
param_grid_dt = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_dt = GridSearchCV(dt_model, param_grid_dt, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_dt.fit(X_train, y_train)
best_dt = grid_dt.best_estimator_
dt_pred = best_dt.predict(X_test)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
print("Decision Tree Best Params:", grid_dt.best_params_)
print("Decision Tree RMSE:", dt_rmse)


ada_model = AdaBoostRegressor(random_state=42)
param_grid_ada = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 1.0]
}

grid_ada = GridSearchCV(ada_model, param_grid_ada, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_ada.fit(X_train, y_train)
best_ada = grid_ada.best_estimator_
ada_pred = best_ada.predict(X_test)
ada_rmse = np.sqrt(mean_squared_error(y_test, ada_pred))
print("AdaBoost Best Params:", grid_ada.best_params_)
print("AdaBoost RMSE:", ada_rmse)


et_model = ExtraTreesRegressor(random_state=42)
param_grid_et = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_et = GridSearchCV(et_model, param_grid_et, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_et.fit(X_train, y_train)
best_et = grid_et.best_estimator_
et_pred = best_et.predict(X_test)
et_rmse = np.sqrt(mean_squared_error(y_test, et_pred))
print("Extra Trees Best Params:", grid_et.best_params_)
print("Extra Trees RMSE:", et_rmse)


print("\nModel Comparison:")
print("---------------------")
print(f"KNN RMSE: {knn_rmse:.2f}")
print(f"XGBoost RMSE: {xgb_rmse:.2f}")
print(f"Decision Tree RMSE: {dt_rmse:.2f}")
print(f"AdaBoost RMSE: {ada_rmse:.2f}")
print(f"Extra Trees RMSE: {et_rmse:.2f}")


from sklearn.metrics import mean_absolute_error

print("\nAdditional Metrics:")
print("KNN MAE:", mean_absolute_error(y_test, best_knn.predict(X_test)))
print("XGBoost MAE:", mean_absolute_error(y_test, best_xgb.predict(X_test)))
print("Decision Tree MAE:", mean_absolute_error(y_test, best_dt.predict(X_test)))
print("AdaBoost MAE:", mean_absolute_error(y_test, best_ada.predict(X_test)))
print("Extra Trees MAE:", mean_absolute_error(y_test, best_et.predict(X_test)))

Trains and evaluates five regression models (KNN, XGBoost, Decision Tree, AdaBoost, Extra Trees) to predict customer lifetime value using GridSearchCV for hyperparameter tuning. After training, the script compares model performance using RMSE and MAE metrics on the test set.

In [ ]:
import json, itertools
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics         import (silhouette_score,
                                     mean_squared_error,
                                     mean_absolute_error,
                                     r2_score)

from sklearn.cluster         import KMeans
from sklearn.ensemble        import (RandomForestRegressor,
                                     GradientBoostingRegressor)
from xgboost                 import XGBRegressor



user_path   = '/content/drive/MyDrive/FYP/Data/yelp_academic_dataset_user.json'
review_path = '/content/drive/MyDrive/FYP/Data/yelp_academic_dataset_review.json'
MAX_LINES   = 200_000
MIN_REVIEWS = 5
K_FINAL     = 4


def read_json_lines(path, max_lines=None):
    """
    Parameters
    ----------
    path : str
    max_lines : int | None
        If None loads entire file; otherwise loads first `max_lines`.
    """
    with open(path, 'r', encoding='utf8') as handle:
        if max_lines is None:
            records = [json.loads(line) for line in handle]
        else:
            records = [
                json.loads(line)
                for line in itertools.islice(handle, max_lines)
            ]
    return pd.DataFrame.from_records(records)

# ---------- 4. Load ----------
print("Loading user file ...")
user   = read_json_lines(user_path,   MAX_LINES)
print("Rows read (user):", len(user))

print("Loading review file ...")
review = read_json_lines(review_path, MAX_LINES)
print("Rows read (review):", len(review))


active = user[user['review_count'] >= MIN_REVIEWS].copy()

active['total_useful'] = active['useful']
active['total_cool']   = active['cool']
active['total_funny']  = active['funny']
active['avg_rating']   = active['average_stars']

keep_cols = ['user_id','avg_rating',
             'total_useful','total_cool','total_funny',
             'review_count']

df = active[keep_cols].dropna().reset_index(drop=True)


features = ['avg_rating','total_useful','total_cool',
            'total_funny','review_count']

X_scaled = StandardScaler().fit_transform(df[features])

inertia = []
for k in range(2, 11):
    inertia.append(KMeans(n_clusters=k, random_state=42).fit(X_scaled).inertia_)

plt.figure(figsize=(6,3))
plt.plot(range(2,11), inertia, marker='o')
plt.title('Elbow Method for K-Means')
plt.xlabel('k (clusters)'); plt.ylabel('Inertia')
plt.tight_layout(); plt.show()


kmeans        = KMeans(n_clusters=K_FINAL, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)
sil_score     = silhouette_score(X_scaled, df['cluster'])
print(f"\nSilhouette score (k={K_FINAL}): {sil_score:.3f}")

print("\nCluster sizes:")
print(df['cluster'].value_counts().sort_index())

print("\nCluster means:")
cluster_means = df.groupby('cluster')[features].mean().round(2)
print(cluster_means)


target = 'review_count'
X      = df[features].drop(columns=[target])
y      = df[target]

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

models = {
    'RandomForest':  RandomForestRegressor(n_estimators=200,
                                           random_state=42),
    'GradBoost'   :  GradientBoostingRegressor(random_state=42),
    'XGBoost'     :  XGBRegressor(n_estimators=300, learning_rate=0.05,
                                  max_depth=5, random_state=42,
                                  objective='reg:squarederror')
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test)
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE' : mean_absolute_error(y_test, pred),
        'R2'  : r2_score(y_test, pred)
    }

res_df = pd.DataFrame(results).T.round(3)
print("\nModel performance (test set):")
print(res_df)



best_name = res_df['RMSE'].idxmin()
best_mdl  = models[best_name]
importances = pd.Series(best_mdl.feature_importances_, index=X.columns)

plt.figure(figsize=(6,3.5))
importances.sort_values().plot(kind='barh')
plt.title(f'Feature Importance – {best_name}')
plt.tight_layout(); plt.show()


This script performs the full pipeline: it loads Yelp user and review data, filters for active users, engineers key engagement features, and performs K-Means clustering with silhouette validation. It then trains and evaluates three regression models (Random Forest, Gradient Boosting, and XGBoost) to predict customer lifetime value, finally visualizing feature importances from the best-performing model.

In [ ]:
import os, glob

DATA_DIR = "/content/drive/MyDrive/FYP/Data/FYP_CHUNKED"

print("All files in folder:")
for fn in sorted(os.listdir(DATA_DIR)):
    print(" •", fn)

print("\nUser chunks:")
print(glob.glob(os.path.join(DATA_DIR, "user_part*")))

print("\nReview chunks:")
print(glob.glob(os.path.join(DATA_DIR, "review_part*")))

Lists all files in the specified directory and identifies chunked user and review data files using filename patterns. This helps verify that the dataset has been properly split for batch processing.

In [ ]:
import pandas as pd
sample_cols = pd.read_csv(
    "/content/drive/MyDrive/FYP/Data/FYP_CHUNKED/user_part_0.csv",
    nrows=0
).columns.tolist()
print("Columns in user_part_0.csv:\n", sample_cols)

Reads only the column headers from a chunked CSV file (user_part_0.csv) to inspect its structure without loading the full dataset into memory.

In [ ]:
import os, glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    silhouette_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.cluster         import KMeans
from sklearn.ensemble        import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from xgboost                 import XGBRegressor, XGBClassifier


DATA_DIR    = "/content/drive/MyDrive/FYP/Data/FYP_CHUNKED"
MIN_REVIEWS = 5
K_FINAL     = 4
SIL_SAMPLE  = 5000
CHUNK_SIZE  = 100_000


user_files = sorted(glob.glob(os.path.join(DATA_DIR, "user_part_*.csv")))
if not user_files:
    raise FileNotFoundError("No user_part_*.csv files found in " + DATA_DIR)
print("Found user files:", user_files)


chunks = []
for fn in user_files:
    print("→ Streaming JSON-lines from", fn)

    for chunk in pd.read_json(fn, lines=True, chunksize=CHUNK_SIZE):

        keep = []
        if 'user_id'        in chunk: keep.append('user_id')
        if 'review_count'   in chunk: keep.append('review_count')
        if 'useful'         in chunk: keep.append('useful')
        if 'cool'           in chunk: keep.append('cool')
        if 'funny'          in chunk: keep.append('funny')
        if 'average_stars'  in chunk: keep.append('average_stars')

        sub = chunk[keep].copy()


        sub = sub.rename(columns={
            'useful': 'total_useful',
            'cool':   'total_cool',
            'funny':  'total_funny',
            'average_stars': 'avg_rating'
        })


        if 'review_count' in sub:
            sub = sub[sub['review_count'] >= MIN_REVIEWS]
        else:
            continue

        if not sub.empty:
            chunks.append(sub)


df = pd.concat(chunks, ignore_index=True)
print(f"\nTotal active users: {len(df):,}")

features = ['avg_rating','total_useful','total_cool','total_funny','review_count']
X_scaled = StandardScaler().fit_transform(df[features])


inertia = [KMeans(n_clusters=k, random_state=42).fit(X_scaled).inertia_
           for k in range(2,11)]
plt.figure(figsize=(6,3))
plt.plot(range(2,11), inertia, 'o-')
plt.title("Elbow Method for K-Means")
plt.xlabel("k"); plt.ylabel("Inertia")
plt.tight_layout(); plt.show()


kmeans        = KMeans(n_clusters=K_FINAL, random_state=42).fit(X_scaled)
df['cluster'] = kmeans.labels_
sil = silhouette_score(
    X_scaled, df['cluster'],
    sample_size=SIL_SAMPLE, random_state=42
)
print(f"\nSilhouette score (k={K_FINAL}): {sil:.3f}")


print("\nCluster sizes:\n", df['cluster'].value_counts().sort_index())
cm = df.groupby('cluster')[features].mean().round(2)
print("\nCluster means:\n", cm)
plt.figure(figsize=(6,4))
sns.heatmap(cm.T, annot=True, cmap='coolwarm')
plt.title("Cluster Profile Heatmap")
plt.tight_layout(); plt.show()


X, y = df[features].drop(columns=['review_count']), df['review_count']
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42
)

reg_models = {
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GradBoost'   : GradientBoostingRegressor(random_state=42),
    'XGBoost'     : XGBRegressor(
                        n_estimators=300,
                        learning_rate=0.05,
                        max_depth=5,
                        random_state=42,
                        objective='reg:squarederror'
                    )
}
reg_res = {}
for name, m in reg_models.items():
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    reg_res[name] = {
        'RMSE': np.sqrt(mean_squared_error(yte, p)),
        'MAE' : mean_absolute_error(yte, p),
        'R2'  : r2_score(yte, p)
    }
reg_df = pd.DataFrame(reg_res).T.round(3)
print("\nRegression Results:\n", reg_df)
reg_df[['RMSE','MAE','R2']].plot.barh(figsize=(6,4))
plt.title("Regression Model Comparison")
plt.tight_layout(); plt.show()


med    = df['review_count'].median()
y_bin  = (df['review_count'] >= med).astype(int)
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X, y_bin, test_size=0.2, random_state=42
)

clf_models = {
    'KNN'               : KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Random Forest'     : RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost'           : XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'Decision Tree'     : DecisionTreeClassifier(random_state=42),
    'Gradient Boosting' : GradientBoostingClassifier(random_state=42),
    'ADA Boosting'      : AdaBoostClassifier(random_state=42),
    'Extra Trees'       : ExtraTreesClassifier(random_state=42),
}

clf_records = []
for name, clf in clf_models.items():
    clf.fit(Xtr_c, ytr_c)
    ypr = clf.predict(Xte_c)
    clf_records.append({
        'Model'                : name,
        'Training Score'       : clf.score(Xtr_c, ytr_c),
        'Test Score (Accuracy)': accuracy_score(yte_c, ypr),
        'Precision'            : precision_score(yte_c, ypr, zero_division=0),
        'Recall'               : recall_score(yte_c, ypr, zero_division=0),
        'F1 Score'             : f1_score(yte_c, ypr, zero_division=0),
    })

norm_df = pd.DataFrame(clf_records).set_index('Model').round(6)
print("\nNormalized DataFrame:\n", norm_df)


best_reg = reg_df['RMSE'].idxmin()
imp = pd.Series(reg_models[best_reg].feature_importances_, index=features)
plt.figure(figsize=(6,3.5))
imp.sort_values().plot.barh()
plt.title(f"Feature Importance – {best_reg}")
plt.tight_layout(); plt.show()


This full pipeline reads chunked user data, filters and processes it into key features, performs K-Means clustering with Elbow and Silhouette validation, and trains multiple regression and classification models to predict both continuous (CLV) and binary outcomes. It also compares model performance and visualizes feature importances and cluster profiles.

In [ ]:
norm_df[['Test Score (Accuracy)','Precision','Recall','F1 Score']].plot(
    kind='bar', figsize=(9,4)
)
plt.title("Classification Metrics by Model")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

Creates a bar chart comparing classification models based on accuracy, precision, recall, and F1 score to visually assess their relative performance.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)


best_name = norm_df['Test Score (Accuracy)'].idxmax()
best_clf  = clf_models[best_name]
print("Best classifier:", best_name)


ConfusionMatrixDisplay.from_estimator(
    best_clf, Xte_c, yte_c, cmap='Blues', normalize='true'
)
plt.title(f"Confusion Matrix ({best_name})")
plt.show()


y_score = best_clf.predict_proba(Xte_c)[:,1]
fpr, tpr, _ = roc_curve(yte_c, y_score)
auc = roc_auc_score(yte_c, y_score)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve ({best_name})")
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


prec, rec, _ = precision_recall_curve(yte_c, y_score)
ap = average_precision_score(yte_c, y_score)

plt.figure(figsize=(6,4))
plt.plot(rec, prec, label=f"AP = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve ({best_name})")
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

Evaluates the best-performing classification model by plotting its normalized confusion matrix, ROC curve (with AUC), and precision-recall curve (with average precision score) to visualize its predictive quality and class separation ability.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster      import KMeans
from sklearn.metrics      import silhouette_score


CLEAN_DIR  = "/content/drive/MyDrive/FYP/Data/FYP_CHUNKED/cleaned_users"
K_FINAL    = 4
SIL_SAMPLE = 5000


clean_files = sorted(glob.glob(f"{CLEAN_DIR}/*_clean.csv"))
if not clean_files:
    raise FileNotFoundError("No cleaned user CSVs found in " + CLEAN_DIR)

dfs = [pd.read_csv(f) for f in clean_files]
df  = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df):,} active users from {len(clean_files)} files")


features = ['avg_rating','total_useful','total_cool','total_funny','review_count']
X = StandardScaler().fit_transform(df[features])


inertia = []
for k in range(2, 11):
    inertia.append(KMeans(n_clusters=k, random_state=42).fit(X).inertia_)

plt.figure(figsize=(6,3))
plt.plot(range(2,11), inertia, 'o-')
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters k")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()


kmeans     = KMeans(n_clusters=K_FINAL, random_state=42).fit(X)
df['cluster'] = kmeans.labels_
sil = silhouette_score(X, df['cluster'], sample_size=SIL_SAMPLE, random_state=42)
print(f"\nSilhouette score (k={K_FINAL}): {sil:.3f}")


sizes = df['cluster'].value_counts().sort_index()
print("\nCluster sizes:")
print(sizes.to_string())

profiles = df.groupby('cluster')[features].mean().round(2)
print("\nCluster profiles (means):")
print(profiles)


plt.figure(figsize=(6,4))
sns.heatmap(profiles.T, annot=True, cmap='coolwarm')
plt.title("Cluster Profile Heatmap")
plt.tight_layout()
plt.show()

Loads pre-cleaned user data from multiple CSV files, performs K-Means clustering with Elbow and Silhouette validation, and visualizes the resulting user segment profiles using a heatmap. This helps interpret behavioral differences between clusters.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    accuracy_score, precision_score,
    recall_score, f1_score
)
import pandas as pd


features = ['avg_rating','total_useful','total_cool','total_funny']
X_clf = df[features]
y_bin = (df['review_count'] >= df['review_count'].median()).astype(int)


Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X_clf, y_bin, test_size=0.2, random_state=42
)


from sklearn.neighbors    import KNeighborsClassifier
from sklearn.ensemble     import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from sklearn.tree         import DecisionTreeClassifier
from xgboost              import XGBClassifier

clf_models = {
    'KNN'               : KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Random Forest'     : RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost'           : XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'Decision Tree'     : DecisionTreeClassifier(random_state=42),
    'Gradient Boosting' : GradientBoostingClassifier(random_state=42),
    'ADA Boosting'      : AdaBoostClassifier(random_state=42),
    'Extra Trees'       : ExtraTreesClassifier(random_state=42),
}

# 4. Train each and collect metrics
records = []
for name, clf in clf_models.items():
    clf.fit(Xtr_c, ytr_c)
    train_score = clf.score(Xtr_c, ytr_c)
    y_pred      = clf.predict(Xte_c)

    records.append({
        'Model'                 : name,
        'Training Score'        : train_score,
        'Test Score (Accuracy)' : accuracy_score(yte_c, y_pred),
        'Precision'             : precision_score(yte_c, y_pred, zero_division=0),
        'Recall'                : recall_score(yte_c, y_pred, zero_division=0),
        'F1 Score'              : f1_score(yte_c, y_pred, zero_division=0)
    })


norm_df = pd.DataFrame(records).set_index('Model').round(6)

print("Normalized DataFrame:")
display(norm_df)

Trains multiple classification models to predict whether a user’s review count is above the median, using engagement features. It evaluates each model on accuracy, precision, recall, and F1 score, compiling the results into a normalized DataFrame for comparison.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import time


rand_grids = {
    'KNN': (
        KNeighborsClassifier(),
        {
            'n_neighbors': [3,5,7,9,11],
            'weights':     ['uniform','distance'],
            'algorithm':   ['auto','ball_tree','kd_tree']
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {
            'n_estimators': randint(25,200),
            'max_depth':    randint(3,20),
            'criterion':    ['gini','entropy']
        }
    ),
    'XGBoost': (
        XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42,
                      early_stopping_rounds=5),
        {
            'n_estimators':   randint(50,300),
            'max_depth':      randint(3,10),
            'learning_rate':  uniform(0.01,0.19),
            'subsample':      uniform(0.5,0.5)
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {
            'criterion':        ['gini','entropy'],
            'max_depth':        [None,5,10,15],
            'min_samples_split':[2,5,10]
        }
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=42),
        {
            'n_estimators':   randint(50,200),
            'learning_rate':  uniform(0.01,0.19),
            'max_depth':      randint(3,10),
            'subsample':      uniform(0.5,0.5)
        }
    ),
    'ADA Boosting': (
        AdaBoostClassifier(random_state=42),
        {
            'n_estimators':   randint(50,200),
            'learning_rate':  uniform(0.01,1.0)
        }
    ),
    'Extra Trees': (
        ExtraTreesClassifier(random_state=42),
        {
            'n_estimators':   randint(50,200),
            'max_depth':      randint(3,20),
            'criterion':      ['gini','entropy']
        }
    )
}


n_iter = 10
cv     = 2
rand_searches = {}

start = time.time()
for name, (estimator, dist) in rand_grids.items():
    print(f"\n=== Randomized search for {name} ===")
    rs = RandomizedSearchCV(
        estimator,
        dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    if name == 'XGBoost':
        rs.fit(Xtr_c, ytr_c, eval_set=[(Xte_c, yte_c)], verbose=False)
    else:
        rs.fit(Xtr_c, ytr_c)

    rand_searches[name] = rs
    print(f"→ Best params for {name}: {rs.best_params_}")

duration = time.time() - start
print(f"\nTotal tuning time: {duration/60:.1f} minutes")


Performs hyperparameter tuning for multiple classification models using RandomizedSearchCV. For each model, it searches over a predefined distribution of hyperparameters to find the best combination based on accuracy, storing results and reporting the optimal configuration and total tuning time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA


features = ['avg_rating',
            'total_useful',
            'total_cool',
            'total_funny',
            'review_count']

X         = df[features]
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)


pca         = PCA(n_components=2, random_state=42)
components  = pca.fit_transform(X_scaled)

plot_df             = df.copy()
plot_df['PC1']      = components[:, 0]
plot_df['PC2']      = components[:, 1]


plt.figure(figsize=(10, 6))
sns.scatterplot(data=plot_df,
                x='PC1', y='PC2',
                hue='cluster',
                palette='Set2',
                s=20, alpha=0.5)

plt.title('PCA Scatter Plot of Yelp User Clusters')
plt.xlabel(f'PC 1  ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC 2  ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


Performs Principal Component Analysis (PCA) to reduce the high-dimensional user feature space to two components for visualization. The resulting scatter plot shows how user clusters are distributed in the reduced space, helping to visually assess cluster separation.

In [ ]:
from sklearn.metrics import silhouette_samples
import matplotlib.pyplot as plt
import numpy as np

# X_scaled = your standardized data
# labels     = kmeans.labels_

sil_vals = silhouette_samples(X_scaled, labels)
k = labels.max() + 1
y_lower = 10

plt.figure(figsize=(6,4))
for i in range(k):
    ith_vals = sil_vals[labels == i]
    ith_vals.sort()
    size = ith_vals.shape[0]
    y_upper = y_lower + size
    color = plt.cm.Set2(i / k)
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals,
                      facecolor=color, edgecolor=color, alpha=0.7)
    plt.text(-0.05, (y_lower + y_upper) / 2, str(i))
    y_lower = y_upper + 10  # 10-pixel gap between clusters

plt.axvline(sil_vals.mean(), color="red", linestyle="--",
            label=f"Mean = {sil_vals.mean():.3f}")
plt.xlabel("Silhouette coefficient")
plt.ylabel("Cluster label")
plt.title("Silhouette plot for k = 4")
plt.legend()
plt.tight_layout()
plt.show()